In [ ]:
import os

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, random_split

In [ ]:
data_base_dir = "/data/CIFAR-10-C"

if not os.path.exists(data_base_dir):
    candidate = "./data/CIFAR-10-C"
    data_base_dir = candidate if os.path.exists(candidate) else "./playground/data/CIFAR-10-C"

print("Using data dir:", data_base_dir)

In [ ]:
x_path = os.path.join(data_base_dir, "gaussian_noise.npy")
y_path = os.path.join(data_base_dir, "labels.npy")

X = np.load(x_path)     # expected shape: (50000, 32, 32, 3)
y = np.load(y_path)     # expected shape: (50000,)

print("X shape:", X.shape, "dtype:", X.dtype)
print("y shape:", y.shape, "dtype:", y.dtype)
print("num classes:", len(np.unique(y)))

In [ ]:
X_tensor = torch.tensor(X, dtype=torch.float32).permute(0, 3, 1, 2) / 255.0  # (N, 3, 32, 32)
y_tensor = torch.tensor(y, dtype=torch.long)

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),                             # 16x16
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),                             # 8x8
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 256), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleCNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

In [ ]:
def train(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct = 0, 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y_batch)
        correct += (logits.argmax(1) == y_batch).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct = 0, 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            logits = model(X_batch)
            total_loss += criterion(logits, y_batch).item() * len(y_batch)
            correct += (logits.argmax(1) == y_batch).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

In [ ]:
# ── CONFORMAL PREDICTION ──────────────────────────────────────────────────────
import math

ALPHA = 0.1

# 1. split: train / calibration / test
dataset = TensorDataset(X_tensor, y_tensor)
n = len(dataset)
train_size = int(0.6 * n)
cal_size   = int(0.2 * n)
test_size  = n - train_size - cal_size

generator = torch.Generator().manual_seed(42)
train_ds, cal_ds, test_ds = random_split(dataset, [train_size, cal_size, test_size], generator)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
cal_loader   = DataLoader(cal_ds,   batch_size=64)
test_loader  = DataLoader(test_ds,  batch_size=64)

In [ ]:
# 2. (Re-)train model
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
for epoch in range(10):
    train_loss, train_acc = train(model, train_loader, optimizer, criterion)
    val_loss, val_acc     = evaluate(model, cal_loader, criterion)
    print(f"Epoch {epoch+1:2d} | Train {train_loss:.4f}/{train_acc:.3f} | Cal {val_loss:.4f}/{val_acc:.3f}")

In [ ]:
# 3. Calibration – collect nonconformity scores s_i = 1 - ŷ[true_class]
model.eval()
cal_scores = []
with torch.no_grad():
    for X_batch, y_batch in cal_loader:
        probs = torch.softmax(model(X_batch.to(device)), dim=1).cpu()
        true_class_probs = probs[torch.arange(len(y_batch)), y_batch]
        cal_scores.append(1.0 - true_class_probs)

cal_scores = torch.cat(cal_scores)          # shape: (n_cal,)

# Finite-sample corrected quantile level (Angelopoulos & Bates, 2022, Section 1.1, Step 3)
n_cal = len(cal_scores)
k = min(math.ceil((n_cal + 1) * (1 - ALPHA)), n_cal)  # number of scores to include in the prediction set
q_hat = torch.sort(cal_scores).values[k - 1]  # q̂ is the k-th smallest score
print(f"\nCalibration quantile q̂ = {q_hat:.4f}  (\N{greek small letter alpha}={ALPHA}, n_cal={n_cal})")

In [ ]:
# 4. Test – build prediction sets and measure coverage / efficiency
model.eval()
covered, total, set_sizes = 0, 0, []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        probs = torch.softmax(model(X_batch.to(device)), dim=1).cpu()

        s_i = torch.tensor(np.ones_like(probs)) - probs  # shape: (batch_size, num_classes)
        pred_sets = s_i <= q_hat  # shape: (batch_size, num_classes)

        for i, y_true in enumerate(y_batch):
            in_set = pred_sets[i, y_true.item()].item()
            covered += int(in_set)
            set_sizes.append(pred_sets[i].sum().item())
        total += len(y_batch)

empirical_coverage = covered / total
avg_set_size       = sum(set_sizes) / len(set_sizes)
print(f"Empirical coverage : {empirical_coverage:.4f}  (target ≥ {1-ALPHA:.2f})")
print(f"Avg prediction-set size: {avg_set_size:.3f}  (out of 10 classes)")

In [ ]:
# 5. Per-class coverage
classes = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']
print("\nPer-class coverage:")
class_covered = [0] * 10
class_total   = [0] * 10
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        probs = torch.softmax(model(X_batch.to(device)), dim=1).cpu()
        s_i = torch.tensor(np.ones_like(probs)) - probs
        pred_sets = s_i <= q_hat
        for i, y_true in enumerate(y_batch):
            c = y_true.item()
            class_covered[c] += int(pred_sets[i, c].item())
            class_total[c]   += 1
for i, cls in enumerate(classes):
    print(f"  {cls:<12}: {class_covered[i]/class_total[i]:.3f}")